# Visualizing evaluation results

Exploring the Parquet files generated by Docling-eval

In [ ]:
from pathlib import Path
from pyspark.sql import SparkSession
from docling_core.types import DoclingDocument
from IPython.display import display
from docling_core.transforms.visualizer.table_visualizer import TableVisualizer
from cells2table.utils.eval.main import analyze_image, pil_to_cv2

%matplotlib widget

Creating a SparkSession to read Parquet files

In [ ]:
spark = SparkSession.builder.appName("cells2table").getOrCreate()

## DoclingDPBench

Bad ground-truth annotations:
- Missing table (doc_eded7804127acce26e4b9e7138aa8c95d0d8886280a21148efb0ae9519941ede_page_000001.png_layout.html)
- Bad table structure (doc_39f32d6a01b8e434ad2fb16ff3896931153cdf4f85bca09194b71e1514711355_page_000001.png_layout.html)
- Missing "PORT" cell text (doc_bdf0e09acee7a07f7e60c6055128967b36b44564842983ad3b0f273f1c1c5914_page_000001.png_layout.html)
- Bad table structure (doc_4d5b682effaf9580928a7073980e3916687d168238bc96b6c17222f747eacde6_page_000001.png_layout.html)
- Really bad in general (doc_770c9434a3530bfb67ef32510b63e1da347dfd6c51e5db9e0b13e90c8b287895_page_000001.png_layout.html)

Bad cells2table results:
- Tolerance fail (doc_bdf0e09acee7a07f7e60c6055128967b36b44564842983ad3b0f273f1c1c5914_page_000001.png_layout.html)
- Tolerance fail (doc_b9e5879b4dcd812e854cc06c6c3cb4f54231af28c919be63e27bef4c639551c9_page_000001.png_layout.html)
- row_span fail (doc_646f014ab615ac22266a43be2691010032d9fd62368737d488ffcb912cd68fd2_page_000001.png_layout.html)
- col_span fail (doc_c31d842c665085cb1379d441ef0b65869f29537b9252956d6105e6b4cc972311_page_000001.png_layout.html)

Reading the prediction files

In [ ]:
benchmark_path = Path("../benchmarks/DoclingDPBench/cells2table/test/")
df = spark.read.parquet(str(benchmark_path.resolve()))

Understanding the Parquet schema

In [ ]:
df.printSchema(level=1)

Extracting the DoclingDocument of a prediction

In [ ]:
# The ID of the document we want to analyze
doc_id = "doc_4d5b682effaf9580928a7073980e3916687d168238bc96b6c17222f747eacde6_page_000001.png"

row = df.filter(df["document_id"] == doc_id).first()
if row is None:
    raise ValueError("Document not found")

doc = DoclingDocument.model_validate_json(row["PredictedDocument"])

Visualizing the prediction

In [ ]:
table = doc.tables[0]
table_img = table.get_image(doc)
if table_img is None:
    raise ValueError("Document generated without generate_page_images=True")

table_visualizer = TableVisualizer()
pred_img = table_visualizer.get_visualization(doc=doc)[1]
display(pred_img)

print(table.export_to_html(doc))

Running the table image with the pipeline (again) to analyze cell confidence scores

That information is lost when building a DoclingDocument

In [ ]:
analyze_image(pil_to_cv2(table_img))

Now run an specific detection model directly, with no classifier

- 0: Wired
- 1: Wireless


In [ ]:
analyze_image(pil_to_cv2(table_img), detection_model_idx=1)